# NPHCDA Zero-Dose - LGA-level archetype modelling (reproducible)
**Consortium: CIDRE and Quantium Insights LLC, in technical support of NPHCDA.**

This notebook reproduces the local-government (LGA, admin-2) archetype analysis for all 774 LGAs:
build a covariate master from published admin-2 surfaces, cluster with **agglomerative hierarchical
clustering (Ward linkage)** - the same method used at state level - join the modelled zero-dose, and
produce the overlay map, Domain 2 (dropout drivers) and Domain 3 (equity index) updates.

**Predictors (15):** education, stunting, wasting, underweight, DPT1-DPT3 dropout, poverty, exclusive
breastfeeding, oral rehydration solution coverage, antenatal care 4+, facility delivery, improved
water, travel time to care, Relative Wealth Index, political-violence events and fatalities.
**Outcome:** modelled zero-dose (joined after clustering, never a clustering input).

> Data are the IHME / DHS / Meta / Weiss / ACLED admin-2 layers (download per the report's data table)
> plus the platform's LGA population reference and GRID3 geometry. Set `BASE` below to the folder that
> holds the covariate subfolders and `LGA_archetype_master.xlsx`.

## 0. Setup

In [ ]:
# In Colab: pip install and mount Drive, then point BASE at the data folder.
# !pip -q install pandas geopandas rasterio rasterstats openpyxl scikit-learn scipy matplotlib mgwr
import os
BASE = "."   # folder containing the covariate subfolders + LGA_archetype_master.xlsx
os.chdir(BASE)
print("working dir:", os.getcwd())

## 1. Build the covariate master (774 LGAs x 15 predictors)
Registry-driven: each covariate is one entry (CSV, raster or point file). Bridges every layer to the
platform's canonical (State, LGA) so heterogeneous IHME/DHS/ACLED products align. Run once.

In [ ]:
"""
Build the LGA-level archetype MASTER dataset (Nigeria, admin-2)
==============================================================
Combines the IHME LMIC admin-2 (LGA) covariate layers you download into ONE clean
matrix, ready to later join to the zero-dose data for the LGA archetype analysis.

Why this is robust: every IHME LMIC admin-2 file shares the same ADM2_CODE, so the
covariate layers join to each other EXACTLY (no fuzzy matching between layers). A single
name-bridge step then maps each LGA to the platform's canonical (State, LGA) so the
zero-dose table can be merged in afterwards.

To add a new covariate: download its admin-2 CSV, drop the folder in, and add ONE line
to COVARIATES below. Re-run. That's it.

Output: LGA_archetype_master.xlsx
  - "LGA covariates"  : one row per LGA, all covariates, plus the canonical join keys
  - "Sources"         : each covariate's file, measure, year, unit and direction
  - "Unmatched bridge": IHME LGAs not yet mapped to a platform LGA name (reconcile by hand)
  - "Join guide"      : how to connect your zero-dose data
Needs: pandas, openpyxl.
"""
from __future__ import annotations
import difflib
import re
import pandas as pd

YEAR = 2017          # latest year in the IHME LMIC 2000-2017 records
COUNTRY = "Nigeria"

# ---- REGISTRY: add one row per covariate as you add folders ---------------------------
# name        -> column name in the master
# file        -> path to the admin-2 (AD2 / ADMIN_2) CSV
# definition  -> plain-language definition
# higher_is   -> "worse" or "better" (for the archetype interpretation; not used in maths)
D = "Data [CSV]"
COVARIATES = [
    {"name": "edu_mean_years_women_15_49",
     "file": f"Education Attainment/{D}/IHME_LMIC_EDU_2000_2017_MEAN_15_49_FEMALE_AD2_Y2019M12D24.CSV",
     "definition": "Mean years of schooling, women 15-49", "higher_is": "better"},
    {"name": "stunting_prev_u5",
     "file": f"Child growth/{D}/IHME_LMIC_CGF_2000_2017_STUNTING_PREV_ADMIN_2_Y2020M01D08.CSV",
     "definition": "Stunting prevalence, children under 5", "higher_is": "worse"},
    {"name": "wasting_prev_u5",
     "file": f"Child growth/{D}/IHME_LMIC_CGF_2000_2017_WASTING_PREV_ADMIN_2_Y2020M01D08.CSV",
     "definition": "Wasting prevalence, children under 5", "higher_is": "worse"},
    {"name": "underweight_prev_u5",
     "file": f"Child growth/{D}/IHME_LMIC_CGF_2000_2017_UNDERWEIGHT_PREV_ADMIN_2_Y2020M01D08.CSV",
     "definition": "Underweight prevalence, children under 5", "higher_is": "worse"},
    # DPT1-to-DPT3 dropout is a DISTINCT failure mode (reached but not completed) -> a valid predictor.
    {"name": "dpt1_3_dropout",
     "file": "DPT Vaccine Coverage/IHME_AFRICA_DPT_2000_2016_REL_DROPOUT_PREV_ADMIN_2_Y2019M04D01.CSV",
     "definition": "DPT1-to-DPT3 relative dropout (reached but not completed)",
     "higher_is": "worse", "role": "predictor", "year": 2016, "encoding": "latin-1"},
    # Equity + care-seeking predictors
    {"name": "poverty_rate", "file": "Poverty/povertyrate.csv",
     "adm0_col": "NAME_0", "adm1_col": "NAME_1", "adm2_col": "NAME_2", "value_col": "Poverty_Ra",
     "year": None, "year_label": 2014,
     "definition": "Poverty headcount rate (~2014)", "higher_is": "worse", "role": "predictor"},
    {"name": "exclusive_breastfeeding",
     "file": "Exclusive Breastfeeding/IHME_AFRICA_EBF_2000_2017_PREV_ADMIN_2_Y2019M07D01.CSV",
     "definition": "Exclusive breastfeeding prevalence (maternal care behaviour proxy)",
     "higher_is": "better", "role": "predictor", "year": 2017},
    {"name": "ors_coverage",
     "file": "Oral Rehydration Therapy/IHME_LMIC_ORT_2000_2017_ORS_ADMIN_2_Y2020M05D27.CSV",
     "definition": "Oral rehydration solution coverage (care-seeking / treatment proxy)",
     "higher_is": "better", "role": "predictor", "year": 2017, "metric": "Percent"},
    # Service-coverage predictors from DHS 2018 modelled-surface rasters (zonal mean onto LGAs)
    {"name": "anc4plus", "file": "ANC4+ Spatial files/NG2018DHS_RHANCNWN4P_MS_MEAN_v01.tif",
     "definition": "ANC 4+ visits, proportion (DHS 2018 modelled surface)", "higher_is": "better",
     "role": "predictor", "year_label": 2018, "measure_label": "ANC 4+ visits"},
    {"name": "delivery_hf", "file": "Delivery HF Spatial files/NG2018DHS_RHDELPCDHF_MS_MEAN_v01.tif",
     "definition": "Delivery in a health facility, proportion (DHS 2018 modelled surface)",
     "higher_is": "better", "role": "predictor", "year_label": 2018, "measure_label": "Facility delivery"},
    {"name": "improved_water",
     "file": "Population using an improved water source/NG2018DHS_WSSRCEPIMP_MS_MEAN_v01.tif",
     "definition": "Population using an improved water source, proportion (DHS 2018 surface)",
     "higher_is": "better", "role": "predictor", "year_label": 2018, "measure_label": "Improved water source"},
    # Access / remoteness and wealth
    {"name": "travel_time_hc",
     "file": "Motorized travel times/2020_motorized_travel_time_to_healthcare.geotiff",
     "definition": "Motorized travel time to healthcare, minutes (Weiss 2020)",
     "higher_is": "worse", "role": "predictor", "year_label": 2020,
     "measure_label": "Travel time to healthcare (min)"},
    {"name": "relative_wealth_index", "file": "Relative wealth index/nga_relative_wealth_index.csv",
     "kind": "points", "lat_col": "latitude", "lon_col": "longitude", "value_col": "rwi",
     "definition": "Relative Wealth Index (Meta), LGA mean", "higher_is": "better",
     "role": "predictor", "year_label": 2021, "measure_label": "Relative Wealth Index"},
    # Insecurity: ACLED political-violence events + fatalities per LGA, 2021-2024 (log-transformed
    # in the clustering step because they are highly right-skewed).
    {"name": "conflict_events", "file": "Political violence ACLED/acled_pv_lga_2021_2024.csv",
     "adm1_col": "Admin1", "adm2_col": "Admin2", "value_col": "conflict_events", "year": None,
     "year_label": "2021-2024", "definition": "ACLED political-violence events (2021-2024)",
     "higher_is": "worse", "role": "predictor", "fillna": 0},
    {"name": "conflict_fatalities", "file": "Political violence ACLED/acled_pv_lga_2021_2024.csv",
     "adm1_col": "Admin1", "adm2_col": "Admin2", "value_col": "conflict_fatalities", "year": None,
     "year_label": "2021-2024", "definition": "ACLED political-violence fatalities (2021-2024)",
     "higher_is": "worse", "role": "predictor", "fillna": 0},
    # NOT added: U-5 mortality (a health OUTCOME, not an upstream driver) and the LLS 2018-2019
    # microdata (household/community survey, not admin-2 and not LGA-representative). See chat notes.
    # NOTE: DPT1 / DPT3 / MCV1 COVERAGE are deliberately EXCLUDED. Vaccine coverage equals zero-dose
    # inverted, so it must not be a clustering input (circular). The archetype OUTCOME is the
    # platform's own modelled zero-dose at LGA level, joined separately on platform_State/platform_LGA.
    # The coverage CSVs remain in the folders and can be enabled later for a separate validation
    # cross-check if wanted (re-add with "role": "outcome").
]

LGA_REFERENCE_CSV = r"..\claude_model\data\sample\administrative_lga_population.csv"
REF_STATE, REF_LGA, REF_STATUS = "State", "Name", "Status"
OUT_XLSX = "LGA_archetype_master.xlsx"

# For raster (.tif) layers (e.g. DHS Spatial Data Repository surfaces): the 774-LGA geometry used
# to compute the zonal mean. Its state/lga fields are already platform-canonical names.
LGA_GEOJSON = r"..\claude_model\data\sample\geo\nga_lgas.geojson"
GEO_STATE, GEO_LGA = "state", "lga"

# LGA renames (normalised IHME spelling -> platform name). Containment logic handles combined
# names like "Yewa North (Egbado North)" and 8-char truncations; these cover the rest. The alias
# is SAFE: it only assigns when the target resolves to a unique LGA within the state.
IHME_ALIAS = {
    "egbadonorth": "Yewa North", "egbadosouth": "Yewa South", "gboyin": "Aiyekire",
    "girie": "Girei", "teungo": "Toungo", "gamjuwa": "Ganjuwa", "yenegoa": "Yenagoa",
    "katsinabenue": "Katsina-Ala", "ahizumb": "Ahiazu-Mbaise",
    "kotonkar": "Kogi", "kontogur": "Kontagora", "borsari": "Bursari",
    "odo0tin": "Odo-Otin", "kano": "Kano Municipal", "arewa": "Arewa Dandi",
    "abohmba": "Abo-Mbaise", "obomangwa": "Obi Ngwa", "dankowasagu": "Wasagu/Danko",
    # Africa-DPT product spellings (differ from the LMIC files)
    "bukuru": "Buruku", "egbadoyewanorth": "Yewa North", "egbadoyewasouth": "Yewa South",
    "kotonkarfe": "Kogi",
    # ACLED political-violence spellings
    "abadan": "Abadam", "danko": "Wasagu/Danko", "minna": "Chanchaga", "kiyawa": "Birnin Magaji",
}
# GADM polygons that are not real LGAs
ARTEFACTS = "Lake Chad|Disputed"

# IHME (LMIC) combines these LGA pairs into one admin-2 surface; both children share the value.
COMBINED_PAIRS = [
    ("Cross River", "Calabar Municipal", "Calabar South"),
    ("Ebonyi", "Afikpo North", "Afikpo South"),
    ("Ekiti", "Emure", "Ise/Orun"),
    ("Ondo", "Ilaje", "Ese-Odo"),
]
STATE_ALIAS = {"nassarawa": "nasarawa", "fct": "federalcapitalterritory",
               "abuja": "federalcapitalterritory"}


def norm(s):
    s = str(s).lower().strip()
    s = re.sub(r"local government area|lga|l\.g\.a\.?|state|municipal area council", "", s)
    return re.sub(r"[^a-z0-9]", "", s)


def _valcol(df):
    return "mean" if "mean" in df.columns else ("value" if "value" in df.columns else None)


def _load_raster(cov):
    """Zonal-mean a .tif surface (e.g. a DHS modelled surface) onto the 774-LGA geometry.
    The geojson's state/lga are already platform-canonical, so bridging is essentially exact."""
    import geopandas as gpd
    import rasterio
    from rasterstats import zonal_stats
    gdf = gpd.read_file(LGA_GEOJSON)
    with rasterio.open(cov["file"]) as r:
        nod = r.nodata
    # all_touched: count any pixel touching the polygon, so small urban LGAs (Lagos Island, etc.)
    # that contain no pixel centroid still get a value from the coarse 5-km surface.
    zs = zonal_stats(gdf, cov["file"], stats=["mean"], nodata=nod, all_touched=True)
    gdf[cov["name"]] = [s["mean"] for s in zs]
    keep = gdf[[GEO_STATE, GEO_LGA, cov["name"]]].rename(
        columns={GEO_STATE: "ADM1_NAME", GEO_LGA: "ADM2_NAME"})
    meta = {"measure": cov.get("measure_label", cov["name"]), "rows": len(keep),
            "value_col": "raster zonal mean", "year": cov.get("year_label", "n/a")}
    return keep, meta


def _load_points(cov):
    """Aggregate a point CSV (e.g. Meta Relative Wealth Index, lat/lon per ~2.4 km cell) to the
    LGA mean via point-in-polygon join onto the 774-LGA geometry."""
    import geopandas as gpd
    df = pd.read_csv(cov["file"])
    lat, lon = cov.get("lat_col", "latitude"), cov.get("lon_col", "longitude")
    val = cov["value_col"]
    pts = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df[lon], df[lat]), crs="EPSG:4326")
    gdf = gpd.read_file(LGA_GEOJSON)[[GEO_STATE, GEO_LGA, "geometry"]]
    j = gpd.sjoin(pts, gdf, how="inner", predicate="within")
    agg = j.groupby([GEO_STATE, GEO_LGA])[val].mean().reset_index()
    keep = agg.rename(columns={GEO_STATE: "ADM1_NAME", GEO_LGA: "ADM2_NAME", val: cov["name"]})
    meta = {"measure": cov.get("measure_label", cov["name"]), "rows": len(keep),
            "value_col": "point-in-polygon mean", "year": cov.get("year_label", "n/a")}
    return keep, meta


def _load_cov(cov):
    """Load one admin-2 layer -> [ADM1_NAME, ADM2_NAME, <name>] for Nigeria. Handles IHME CSVs,
    non-IHME CSVs (overridable columns/year), .tif/.geotiff rasters (zonal mean), and point CSVs
    (kind='points', point-in-polygon mean)."""
    if cov.get("kind") == "points":
        return _load_points(cov)
    if str(cov["file"]).lower().endswith((".tif", ".tiff", ".geotiff")):
        return _load_raster(cov)
    try:
        df = pd.read_csv(cov["file"], encoding=cov.get("encoding"))
    except UnicodeDecodeError:
        df = pd.read_csv(cov["file"], encoding="latin-1")     # some Africa files are latin-1
    a0 = cov.get("adm0_col", "ADM0_NAME")
    a1 = cov.get("adm1_col", "ADM1_NAME")
    a2 = cov.get("adm2_col", "ADM2_NAME")
    v = cov.get("value_col") or _valcol(df)
    d = df.copy()
    if a0 in d.columns:
        d = d[d[a0].astype(str).str.contains(COUNTRY, case=False, na=False)]
    has_year = "year" in d.columns
    yr = cov.get("year", YEAR)
    if has_year and yr is not None:
        d = d[d["year"] == yr]
    # Some files carry the same LGA twice as Percent AND Number - keep the rate, not the count.
    if "metric" in d.columns:
        m = cov.get("metric")
        if m is not None:
            d = d[d["metric"].astype(str).str.lower() == m.lower()]
        elif d["metric"].nunique() > 1:
            d = d[~d["metric"].astype(str).str.contains("Number|Count", case=False, na=False)]
    keep = d[[a1, a2, v]].rename(columns={a1: "ADM1_NAME", a2: "ADM2_NAME", v: cov["name"]})
    meta = {"measure": df["measure"].iloc[0] if "measure" in df.columns else cov["name"],
            "rows": len(keep), "value_col": v,
            "year": (yr if has_year else cov.get("year_label", "n/a"))}
    return keep, meta


def build():
    # Canonical platform LGA spine + bridge setup. Different IHME products (LMIC vs Africa) use
    # different ADM2_CODE schemes, so we bridge EVERY layer by name to the platform (State, LGA)
    # and merge on that key - not on ADM2_CODE.
    ref = pd.read_csv(LGA_REFERENCE_CSV)
    if REF_STATUS in ref.columns:
        ref = ref[ref[REF_STATUS].astype(str).str.strip() == "Local Government Area"]
    ref = ref[~ref[REF_LGA].astype(str).str.contains(ARTEFACTS, case=False, na=False)].copy()
    ref["sk"] = ref[REF_STATE].map(lambda s: STATE_ALIAS.get(norm(s), norm(s)))
    ref["lk"] = ref[REF_LGA].map(norm)

    def _find(pool, key):
        """exact -> unique containment (handles 'Yewa North (Egbado North)' and 8-char truncations)."""
        exact = pool[pool["lk"] == key]
        if len(exact):
            return exact.iloc[0]
        if len(key) >= 6:
            cont = pool[pool["lk"].map(lambda k: key in k or k in key)]
            if cont[REF_LGA].nunique() == 1:
                return cont.iloc[0]
        return None

    _cache = {}

    def bridge(state, lga):
        k = (str(state), str(lga))
        if k in _cache:
            return _cache[k]
        sk = STATE_ALIAS.get(norm(state), norm(state))
        lk = norm(lga)
        sub = ref[ref["sk"] == sk]
        pool = sub if len(sub) else ref
        row = None
        if lk in IHME_ALIAS:
            row = _find(pool, norm(IHME_ALIAS[lk]))
        if row is None:
            row = _find(pool, lk)
        if row is None:
            c = difflib.get_close_matches(lk, pool["lk"].tolist(), n=1, cutoff=0.86)
            row = pool[pool["lk"] == c[0]].iloc[0] if c else None
        out = (row[REF_STATE], row[REF_LGA]) if row is not None else (None, None)
        _cache[k] = out
        return out

    spine = (ref[[REF_STATE, REF_LGA]]
             .rename(columns={REF_STATE: "platform_State", REF_LGA: "platform_LGA"})
             .drop_duplicates().reset_index(drop=True))

    master, sources, unmatched_rows = spine.copy(), [], []
    for cov in COVARIATES:
        layer, meta = _load_cov(cov)
        layer = layer[~layer["ADM2_NAME"].astype(str).str.contains(ARTEFACTS, case=False, na=False)]
        br = [bridge(s, l) for s, l in zip(layer["ADM1_NAME"], layer["ADM2_NAME"])]
        layer["platform_State"] = [b[0] for b in br]
        layer["platform_LGA"] = [b[1] for b in br]
        for _, r in layer[layer["platform_LGA"].isna()].iterrows():
            unmatched_rows.append({"covariate": cov["name"], "State (IHME)": r["ADM1_NAME"],
                                   "LGA (IHME)": r["ADM2_NAME"]})
        t = (layer.dropna(subset=["platform_LGA"])
             .drop_duplicates(["platform_State", "platform_LGA"])
             [["platform_State", "platform_LGA", cov["name"]]])
        master = master.merge(t, on=["platform_State", "platform_LGA"], how="left")
        matched = int(layer["platform_LGA"].notna().sum())
        sources.append({"covariate": cov["name"], "role": cov.get("role", "predictor"),
                        "file": cov["file"].split("/")[-1], "measure": meta["measure"],
                        "year": meta["year"], "unit_col": meta["value_col"],
                        "definition": cov["definition"], "higher_is": cov["higher_is"],
                        "IHME LGAs bridged": f"{matched}/{len(layer)}"})
        print(f"  + {cov['name']:26s} bridged {matched}/{len(layer)}")

    preds = [c["name"] for c in COVARIATES if c.get("role", "predictor") == "predictor"]
    outs = [c["name"] for c in COVARIATES if c.get("role", "predictor") != "predictor"]
    cov_cols = preds + outs
    master = master[["platform_State", "platform_LGA"] + cov_cols].sort_values(
        ["platform_State", "platform_LGA"]).reset_index(drop=True)

    # Share each IHME combined-unit surface value across both child LGAs (symmetric fill).
    mi = master.set_index(["platform_State", "platform_LGA"])
    for st, a, b in COMBINED_PAIRS:
        ia, ib = (st, a), (st, b)
        if ia in mi.index and ib in mi.index:
            for c in cov_cols:
                if pd.isna(mi.at[ia, c]) and pd.notna(mi.at[ib, c]):
                    mi.at[ia, c] = mi.at[ib, c]
                elif pd.isna(mi.at[ib, c]) and pd.notna(mi.at[ia, c]):
                    mi.at[ib, c] = mi.at[ia, c]
    master = mi.reset_index()

    # Per-covariate fill (e.g. ACLED absence = zero recorded violence, not missing).
    for _c in COVARIATES:
        if _c.get('fillna') is not None and _c['name'] in master.columns:
            master[_c['name']] = master[_c['name']].fillna(_c['fillna'])

    miss_mask = master[cov_cols].isna().any(axis=1)
    missing = master[miss_mask][["platform_State", "platform_LGA"] + cov_cols]
    missing = missing.assign(missing_covariates=master[cov_cols].isna().sum(axis=1)[miss_mask])
    coverage = master[cov_cols].notna().all(axis=1).sum()
    print(f"\nMaster: {len(master)} platform LGAs | {len(cov_cols)} covariates "
          f"({len(preds)} predictors, {len(outs)} outcome/validation)")
    print(f"  LGAs with every covariate present: {coverage}/{len(master)} | "
          f"LGAs with >=1 missing covariate: {len(missing)}")

    unmatched = (pd.DataFrame(unmatched_rows).drop_duplicates()
                 if unmatched_rows else pd.DataFrame(columns=["covariate", "State (IHME)", "LGA (IHME)"]))
    guide = pd.DataFrame({"How to connect your zero-dose data": [
        "1. Each row is one canonical Nigeria LGA with its IHME PREDICTOR covariates.",
        "2. OUTCOME: join your modelled zero-dose LGA table onto this sheet using",
        "   platform_State + platform_LGA. Zero-dose is the archetype outcome and is NOT in this",
        "   file by design - it comes from the platform model, not IHME.",
        "3. CLUSTERING: z-score all predictor columns, then cluster; direction (worse/better) on Sources.",
        "4. Vaccine coverage (DPT1/DPT3/MCV1) is deliberately EXCLUDED - it equals zero-dose inverted,",
        "   so it would be circular. Those CSVs stay in the folders for a separate validation if wanted.",
        "5. Add a predictor: drop the folder in, add one line to COVARIATES, re-run.",
        "",
        "Vintage: IHME modelled estimates - education/growth 2017, dropout 2016 (see Sources per layer).",
    ]})

    with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as w:
        master.to_excel(w, sheet_name="LGA covariates", index=False)
        pd.DataFrame(sources).to_excel(w, sheet_name="Sources", index=False)
        unmatched.to_excel(w, sheet_name="Unmatched bridge", index=False)
        missing.to_excel(w, sheet_name="Missing LGAs", index=False)
        guide.to_excel(w, sheet_name="Join guide", index=False)
    print(f"wrote {OUT_XLSX}  ({len(master)} LGAs x {len(cov_cols)} covariates)")
    return master


if __name__ == "__main__":
    build()


## 2. Agglomerative hierarchical clustering (Ward, k = 5)
StandardScaler z-scoring, k validated by silhouette and Calinski-Harabasz, five archetypes to match the
state-level analysis and its intervention bundles. Predictors only.

In [ ]:
"""
LGA archetypes via agglomerative hierarchical clustering (Ward) - the same unsupervised
methodology used at state level (Domain 5-II): StandardScaler z-scoring -> Ward-linkage
AgglomerativeClustering, with k validated by silhouette + Calinski-Harabasz, k=5 selected
to match the state-level archetype count (and its intervention bundles).

Predictors only (zero-dose is the OUTCOME, joined separately - not a clustering input).
Reads LGA_archetype_master.xlsx; writes assignments, profiles and figures.
"""
import matplotlib
matplotlib.use("Agg")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score

MASTER = "LGA_archetype_master.xlsx"
N_CLUSTERS = 5                              # matches the state-level archetype count
LOG_COLS = ["conflict_events", "conflict_fatalities"]   # right-skewed -> log1p before scaling
NAVY, GREEN, GOLD, RED, STEEL = "#1F3B57", "#1C7A3D", "#C8902A", "#C0392B", "#2E6E8E"

m = pd.read_excel(MASTER, "LGA covariates")
src = pd.read_excel(MASTER, "Sources")
preds = [c for c in src["covariate"] if c in m.columns]
worse = dict(zip(src["covariate"], src["higher_is"]))   # direction per predictor

X = m[preds].copy()
n_missing = int(X.isna().any(axis=1).sum())
X = X.fillna(X.median())                                 # impute the few name-quirk LGAs
for c in LOG_COLS:
    if c in X:
        X[c] = np.log1p(X[c])
Xz = StandardScaler().fit_transform(X)

# --- k validation (silhouette + Calinski-Harabasz), same as state level -----------------
ks = range(2, 10)
sil, ch = [], []
for k in ks:
    lab = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(Xz)
    sil.append(silhouette_score(Xz, lab))
    ch.append(calinski_harabasz_score(Xz, lab))

# --- final clustering --------------------------------------------------------------------
labels = AgglomerativeClustering(n_clusters=N_CLUSTERS, linkage="ward").fit_predict(Xz)
m["cluster"] = labels

# order archetypes by a deprivation score (mean z in the "worse" direction) so A1 = highest need
Zdf = pd.DataFrame(Xz, columns=preds)
sign = np.array([1.0 if worse.get(c) == "worse" else -1.0 for c in preds])
Zdf["dep"] = (Zdf[preds].values * sign).mean(axis=1)
Zdf["cluster"] = labels
order = Zdf.groupby("cluster")["dep"].mean().sort_values(ascending=False).index.tolist()
rank = {c: i + 1 for i, c in enumerate(order)}
m["archetype"] = m["cluster"].map(rank)
# z-scored archetype means from Zdf (kept in ORIGINAL row order - avoids the sort misalignment)
Zdf["archetype"] = Zdf["cluster"].map(rank)
zmean = Zdf.groupby("archetype")[preds].mean()
m = m.sort_values(["archetype", "platform_State", "platform_LGA"])

# --- profiles (original-scale means) + short signature ----------------------------------
prof = m.groupby("archetype")[preds].mean().round(3)
prof.insert(0, "n_LGAs", m.groupby("archetype").size())

def signature(a):
    row = zmean.loc[a]
    hi = ", ".join(f"{c}(+{row[c]:.1f})" for c in row.sort_values(ascending=False).head(3).index)
    lo = ", ".join(f"{c}({row[c]:.1f})" for c in row.sort_values().head(3).index)
    return f"high: {hi} | low: {lo}"

sig = pd.DataFrame({"archetype": list(rank.values()),
                    "signature (z-scored)": [signature(a) for a in sorted(rank.values())]}).sort_values("archetype")
states = (m.groupby("archetype")["platform_State"]
          .agg(lambda s: ", ".join(s.value_counts().head(4).index)).rename("top_states"))

# --- outputs -----------------------------------------------------------------------------
m[["archetype", "platform_State", "platform_LGA"] + preds].to_csv("LGA_archetypes.csv", index=False)
prof.join(states).to_csv("LGA_archetype_profiles.csv")
sig.to_csv("LGA_archetype_signatures.csv", index=False)

# cluster-selection figure
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].plot(list(ks), sil, "o-", color=STEEL, lw=2); ax[0].axvline(N_CLUSTERS, color=RED, ls="--")
ax[0].set_title("Silhouette vs k"); ax[0].set_xlabel("k")
ax[1].plot(list(ks), ch, "s-", color=GREEN, lw=2); ax[1].axvline(N_CLUSTERS, color=RED, ls="--")
ax[1].set_title("Calinski-Harabasz vs k"); ax[1].set_xlabel("k")
fig.suptitle("LGA archetypes - optimal cluster selection", fontweight="bold")
fig.tight_layout(); fig.savefig("fig_cluster_selection.png", dpi=150); plt.close(fig)

# dendrogram (Ward)
Z = linkage(Xz, method="ward")
fig, ax = plt.subplots(figsize=(12, 4.5))
dendrogram(Z, truncate_mode="lastp", p=30, ax=ax, color_threshold=Z[-(N_CLUSTERS - 1), 2],
           no_labels=True)
ax.set_title(f"Ward dendrogram ({len(m)} LGAs) - cut at k={N_CLUSTERS}", fontweight="bold")
fig.tight_layout(); fig.savefig("fig_dendrogram.png", dpi=150); plt.close(fig)

# archetype heatmap (z-scored means)
fig, ax = plt.subplots(figsize=(11, 5))
im = ax.imshow(zmean.reindex(sorted(rank.values())).T, aspect="auto", cmap="RdBu_r", vmin=-1.5, vmax=1.5)
ax.set_xticks(range(N_CLUSTERS)); ax.set_xticklabels([f"A{a}\n(n={prof.loc[a,'n_LGAs']})" for a in sorted(rank.values())])
ax.set_yticks(range(len(preds))); ax.set_yticklabels(preds, fontsize=8)
fig.colorbar(im, label="mean z-score"); ax.set_title("LGA archetype signatures (z-scored predictor means)", fontweight="bold")
fig.tight_layout(); fig.savefig("fig_archetype_heatmap.png", dpi=150); plt.close(fig)

print(f"Clustered {len(m)} LGAs into {N_CLUSTERS} archetypes (Ward). {n_missing} LGAs median-imputed.")
print(f"Silhouette at k=5: {sil[3]:.3f} | k that maximises silhouette: {list(ks)[int(np.argmax(sil))]}")
print("\nArchetype sizes and top states:")
print(prof[["n_LGAs"]].join(states).to_string())
print("\nSignatures:")
for _, r in sig.iterrows():
    print(f"  A{r['archetype']}: {r['signature (z-scored)']}")
print("\nWrote: LGA_archetypes.csv, LGA_archetype_profiles.csv, LGA_archetype_signatures.csv,")
print("       fig_cluster_selection.png, fig_dendrogram.png, fig_archetype_heatmap.png")


## 3. Join the modelled zero-dose and finalise the master
Adds the archetype and the modelled zero-dose (rate + count) to the covariate sheet and writes the
`Archetypes` and `Zero-dose (modeled)` tabs. (Generate `lga_modeled_zero_dose.csv` from the Domain 5
model first.)

In [ ]:
"""
Finalize the LGA archetype master: join the cluster/archetype and the modelled zero-dose into
the covariate sheet (so NPHCDA can filter LGAs), and add the 'Zero-dose (modeled)' and
'Archetypes' tabs. Run AFTER build_lga_archetype_master.py and cluster_lga_archetypes.py.
"""
import re
import pandas as pd

XLSX = "LGA_archetype_master.xlsx"
ARCH = "LGA_archetypes.csv"
ZD = "lga_modeled_zero_dose.csv"

# Archetype typology + intervention mapping (aligned to the NPHCDA archetype slide + Gavi/IRMMA)
TYPOLOGY = {
    1: ("Remote Rural / Hard-to-Reach",
        "Lowest maternal education, highest child undernutrition, lowest facility delivery",
        "IBRA microplanning; BHCPF-funded outreach; seasonal scheduling to farming cycles; CBHW deployment"),
    2: ("Conflict-Affected / Nomadic",
        "Highest political-violence fatalities; high dropout; weak ANC and delivery coverage",
        "Security-integrated microplanning; cross-LGA population tracking; mobile vaccination teams; TBA demand outreach"),
    3: ("Riverine / Geographically Isolated",
        "Extreme travel time to care; low improved-water access",
        "Boat-based outreach; community health worker networks; multi-antigen bundling per visit"),
    4: ("Peri-urban / Migrant Dense (transitional)",
        "Moderate on all dimensions; fair care-seeking (ANC, facility delivery)",
        "Ward-level enumeration; migrant-sensitive scheduling; integrate RI fixed posts and mobile teams"),
    5: ("Urban Slums (better-off urban core)",
        "Highest wealth and education, best coverage; residual gaps in informal settlements",
        "Targeted social mobilization; fixed-site + outreach hybrid; private-sector last-mile reach"),
}


def norm(s):
    return re.sub(r"[^a-z0-9]", "", re.sub(r"local government area|lga|state", "", str(s).lower().strip()))


sheets = pd.read_excel(XLSX, sheet_name=None)
cov = sheets["LGA covariates"].copy()
arch = pd.read_csv(ARCH)[["platform_State", "platform_LGA", "archetype"]]
zd = pd.read_csv(ZD)

# archetype -> covariate sheet (exact platform keys)
cov = cov.merge(arch, on=["platform_State", "platform_LGA"], how="left")

# zero-dose -> covariate sheet via normalised (state, lga)
cov["_k"] = cov["platform_State"].map(norm) + "|" + cov["platform_LGA"].map(norm)
zd["_k"] = zd["State"].map(norm) + "|" + zd["LGA"].map(norm)
zdm = zd.drop_duplicates("_k").set_index("_k")
cov["zero_dose_rate_pct"] = cov["_k"].map(zdm["zero_dose_rate_pct"])
cov["zero_dose_children"] = cov["_k"].map(zdm["zero_dose_children"])
cov = cov.drop(columns="_k")

matched_zd = int(cov["zero_dose_children"].notna().sum())
print(f"covariate rows {len(cov)} | archetype joined {cov['archetype'].notna().sum()} | "
      f"zero-dose joined {matched_zd}")

# reorder: keys, archetype, zero-dose, then covariates
front = ["platform_State", "platform_LGA", "archetype", "zero_dose_rate_pct", "zero_dose_children"]
cov = cov[front + [c for c in cov.columns if c not in front]]

# typology columns on the covariate sheet too (nice for filtering)
cov["archetype_type"] = cov["archetype"].map(lambda a: TYPOLOGY.get(a, ("", "", ""))[0] if pd.notna(a) else "")

# ---- Zero-dose (modeled) tab ----
zdt = cov[["platform_State", "platform_LGA", "archetype", "archetype_type",
           "zero_dose_rate_pct", "zero_dose_children"]].copy()
zdt = zdt.sort_values("zero_dose_children", ascending=False, na_position="last")
zdt.insert(0, "zero_dose_burden_rank", range(1, len(zdt) + 1))

# ---- Archetypes tab (one row per cluster, ALL states, typology, intervention) ----
rows = []
for a in sorted(arch["archetype"].dropna().unique()):
    sub = arch[arch["archetype"] == a]
    vc = sub["platform_State"].value_counts()
    states = "; ".join(f"{s} ({n})" for s, n in vc.items())
    zmean = cov.loc[cov["archetype"] == a, "zero_dose_rate_pct"].mean()
    zsum = cov.loc[cov["archetype"] == a, "zero_dose_children"].sum()
    typ, feat, lever = TYPOLOGY[a]
    rows.append({"Cluster": int(a), "Number of LGAs": len(sub), "Type of Cluster": typ,
                 "States (LGAs)": states, "Feature / defining signature": feat,
                 "Mean zero-dose rate (%)": round(zmean, 1) if pd.notna(zmean) else None,
                 "Zero-dose children (sum)": int(zsum) if pd.notna(zsum) else None,
                 "Targeted lever / intervention": lever})
arch_tab = pd.DataFrame(rows)

with pd.ExcelWriter(XLSX, engine="openpyxl") as w:
    cov.to_excel(w, sheet_name="LGA covariates", index=False)
    arch_tab.to_excel(w, sheet_name="Archetypes", index=False)
    zdt.to_excel(w, sheet_name="Zero-dose (modeled)", index=False)
    for name, df in sheets.items():
        if name != "LGA covariates":
            df.to_excel(w, sheet_name=name, index=False)

print("Archetype summary (zero-dose by cluster):")
print(arch_tab[["Cluster", "Type of Cluster", "Number of LGAs", "Mean zero-dose rate (%)",
                "Zero-dose children (sum)"]].to_string(index=False))
print(f"\nwrote {XLSX} with tabs: LGA covariates (+archetype +zero-dose), Archetypes, Zero-dose (modeled)")


## 4. Archetype x zero-dose overlay figure (publication-grade)

In [ ]:
"""
Publication-grade LGA archetype x zero-dose overlay:
  (A) 774-LGA choropleth coloured by the 5 archetypes, with graduated bubbles at LGA centroids
      sized by the number of modelled zero-dose children (colour = type of problem, size = scale).
  (B) Zero-dose rate distribution by archetype (boxplot) + total zero-dose children per archetype.
Reads the enhanced master + the LGA geojson.
"""
import matplotlib
matplotlib.use("Agg")
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

GEO = r"..\claude_model\data\sample\geo\nga_lgas.geojson"
XLSX = "LGA_archetype_master.xlsx"
# severity-ordered categorical palette (dark red = worst, green = best)
COL = {1: "#7A1616", 2: "#D6604D", 3: "#2C7FB8", 4: "#F0C24B", 5: "#1C7A3D"}
TYPE = {1: "Remote Rural / Hard-to-Reach", 2: "Conflict-Affected / Nomadic",
        3: "Riverine / Isolated", 4: "Peri-urban / Migrant Dense", 5: "Urban Slums / better-off"}
NAVY, INK = "#1F3B57", "#27343D"

cov = pd.read_excel(XLSX, "LGA covariates")
g = gpd.read_file(GEO)
g["_k"] = g["state"].str.lower().str.strip() + "|" + g["lga"].str.lower().str.strip()
cov["_k"] = cov["platform_State"].str.lower().str.strip() + "|" + cov["platform_LGA"].str.lower().str.strip()
g = g.merge(cov[["_k", "archetype", "zero_dose_rate_pct", "zero_dose_children"]], on="_k", how="left")
g["archetype"] = g["archetype"].fillna(0).astype(int)

fig = plt.figure(figsize=(15, 8.5))
gs = fig.add_gridspec(2, 2, width_ratios=[2.2, 1], height_ratios=[1, 1], wspace=0.13, hspace=0.32)
axm = fig.add_subplot(gs[:, 0])

# choropleth
for a, c in COL.items():
    g[g["archetype"] == a].plot(ax=axm, color=c, edgecolor="white", linewidth=0.25)
g[g["archetype"] == 0].plot(ax=axm, color="#E5E5E5", edgecolor="white", linewidth=0.25)

# zero-dose bubbles at centroids (area proportional to children)
cent = g.copy()
cent["geometry"] = cent.geometry.representative_point()
cent = cent[cent["zero_dose_children"].fillna(0) > 0]
sizes = np.sqrt(cent["zero_dose_children"]) * 0.55
axm.scatter(cent.geometry.x, cent.geometry.y, s=sizes, facecolor="#11151A", alpha=0.32,
            edgecolor="white", linewidth=0.3, zorder=5)
axm.set_title("Nigeria LGA archetypes and modelled zero-dose burden",
              fontsize=14, fontweight="bold", color=NAVY, loc="left")
axm.axis("off")

arch_leg = [Patch(facecolor=COL[a], edgecolor="white", label=f"{a}. {TYPE[a]}") for a in COL]
arch_leg.append(Patch(facecolor="#E5E5E5", edgecolor="white", label="Not modelled"))
leg1 = axm.legend(handles=arch_leg, title="Archetype (cluster)", loc="lower left",
                  fontsize=8.5, title_fontsize=9, frameon=False)
axm.add_artist(leg1)
bub = [10000, 20000, 30000]
bub_h = [Line2D([0], [0], marker="o", ls="", markerfacecolor="#11151A", alpha=0.32,
                markeredgecolor="white", markersize=np.sqrt(np.sqrt(b) * 0.55), label=f"{b:,}") for b in bub]
axm.legend(handles=bub_h, title="Zero-dose children", loc="lower right", labelspacing=1.6,
           fontsize=8.5, title_fontsize=9, frameon=False, borderpad=1.1)

# (B1) zero-dose rate by archetype - boxplot
axb = fig.add_subplot(gs[0, 1])
data = [cov.loc[cov["archetype"] == a, "zero_dose_rate_pct"].dropna() for a in COL]
bp = axb.boxplot(data, patch_artist=True, widths=0.6, showfliers=False)
for patch, a in zip(bp["boxes"], COL):
    patch.set_facecolor(COL[a]); patch.set_alpha(0.85)
for med in bp["medians"]:
    med.set_color("white"); med.set_linewidth(1.6)
axb.set_xticklabels(list(COL), fontsize=9)
axb.set_ylabel("Zero-dose rate (%)", fontsize=9)
axb.set_title("Zero-dose rate by archetype", fontsize=10.5, fontweight="bold", color=NAVY, loc="left")
axb.spines[["top", "right"]].set_visible(False)

# (B2) total zero-dose children by archetype - bars
axc = fig.add_subplot(gs[1, 1])
tot = cov.groupby("archetype")["zero_dose_children"].sum().reindex(list(COL))
axc.bar(range(len(COL)), tot.values / 1000, color=[COL[a] for a in COL])
for i, v in enumerate(tot.values):
    axc.text(i, v / 1000 + 8, f"{v/1000:.0f}k", ha="center", fontsize=8, color=INK)
axc.set_xticks(range(len(COL))); axc.set_xticklabels(list(COL), fontsize=9)
axc.set_ylabel("Zero-dose children (000s)", fontsize=9)
axc.set_title("Zero-dose burden by archetype", fontsize=10.5, fontweight="bold", color=NAVY, loc="left")
axc.spines[["top", "right"]].set_visible(False)

fig.text(0.5, 0.02, "Archetypes from agglomerative (Ward) clustering of 15 LGA covariates (IHME, DHS, "
         "Meta RWI, Weiss travel time, ACLED). Bubbles = modelled zero-dose children. "
         "Sources: consortium model; GRID3 admin-2 boundaries.",
         ha="center", fontsize=7.6, color="#5B6B79")
fig.savefig("fig_archetype_zerodose_overlay.png", dpi=200, bbox_inches="tight", facecolor="white")
print("wrote fig_archetype_zerodose_overlay.png |", int(cent.shape[0]), "bubbles")


## 5. Domain 2 (LGA level) - antigen-pair dropout and LASSO drivers
Dropout per LGA from routine data (2021-2024); LASSO on the 15 covariates identifies factors associated
with each transition (Penta1->Penta3, Penta1->Measles1, Measles1->Measles2).

In [ ]:
"""
Domain 2 at LGA level: antigen-pair dropout + LASSO drivers.
Dropout computed per LGA from DHIS2 (2021-2024): Penta1->Penta3, Penta1->Measles1, Measles1->Measles2.
LASSO (LassoCV, standardized predictors) on the 15 LGA archetype covariates identifies which factors
are significantly associated with each dropout transition. Replaces the state-level LASSO.
"""
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, r"C:/Users/Amobi Andrew/OneDrive/Desktop/Immunization Modeling Fellowship/In-person Training/Submission Package/ReadyOne/All I Need/claude_model")
import matplotlib
matplotlib.use("Agg")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
import config as C
import names as N
from models.d5_zerodose import prep_dhis2

NAVY, GREEN, GOLD = "#1F3B57", "#1C7A3D", "#C8902A"

# --- LGA dropout from DHIS2 (2021-2024 totals) ---
d = prep_dhis2(pd.read_csv(C.SAMPLE_FILES["dhis2"]))
d["lga"] = d["lga"].astype(str).map(N.clean_lga_name)
d = d[d["year"].between(2021, 2024)]
ANT = ["penta_1_count", "penta_3_count", "measles_1_count", "measles_2_count"]
for c in ANT:
    d[c] = pd.to_numeric(d[c], errors="coerce")
g = d.groupby(["state", "lga"])[ANT].sum().reset_index()
g["dropout_p1p3"] = ((g["penta_1_count"] - g["penta_3_count"]) / g["penta_1_count"] * 100).clip(-20, 100)
g["dropout_p1m1"] = ((g["penta_1_count"] - g["measles_1_count"]) / g["penta_1_count"] * 100).clip(-20, 100)
g["dropout_m1m2"] = ((g["measles_1_count"] - g["measles_2_count"]) / g["measles_1_count"] * 100).clip(-20, 100)

# --- merge with LGA covariates ---
cov = pd.read_excel("LGA_archetype_master.xlsx", "LGA covariates")
import re
def norm(s): return re.sub(r"[^a-z0-9]", "", re.sub(r"local government area|lga|state", "", str(s).lower().strip()))
g["_k"] = g["state"].map(norm) + "|" + g["lga"].map(norm)
cov["_k"] = cov["platform_State"].map(norm) + "|" + cov["platform_LGA"].map(norm)
PRED = [c for c in cov.columns if c not in ("platform_State", "platform_LGA", "_k", "archetype",
        "archetype_type", "zero_dose_rate_pct", "zero_dose_children")]
m = g.merge(cov[["_k"] + PRED], on="_k", how="inner").dropna(subset=PRED)
print(f"LGAs with dropout + covariates: {len(m)}")
print("National dropout (median %):",
      {k: round(m[f'dropout_{k}'].median(), 1) for k in ["p1p3", "p1m1", "m1m2"]})

TARGETS = [("dropout_p1p3", "Penta1 -> Penta3", GOLD),
           ("dropout_p1m1", "Penta1 -> Measles1", "#8E44AD"),
           ("dropout_m1m2", "Measles1 -> Measles2", NAVY)]
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
results = {}
for ax, (tgt, title, col) in zip(axes, TARGETS):
    y = pd.to_numeric(m[tgt], errors="coerce").replace([np.inf, -np.inf], np.nan).values
    mask = np.isfinite(y)
    Xz = StandardScaler().fit_transform(m.loc[mask, PRED])
    y = y[mask]
    las = LassoCV(cv=5, random_state=0, max_iter=20000).fit(Xz, y)
    coef = pd.Series(las.coef_, index=PRED)
    nz = coef[coef.abs() > 1e-6].sort_values(key=abs, ascending=True)
    results[tgt] = coef
    ax.barh(range(len(nz)), nz.values, color=[GREEN if v < 0 else "#C0392B" for v in nz.values])
    ax.set_yticks(range(len(nz))); ax.set_yticklabels(nz.index, fontsize=8)
    ax.axvline(0, color="#888", lw=0.8)
    ax.set_title(f"{title}\nLASSO drivers (alpha={las.alpha_:.2f}, R2={las.score(Xz, y):.2f})",
                 fontsize=10, fontweight="bold", color=NAVY)
    ax.set_xlabel("Standardized LASSO coefficient"); ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("Domain 2 (LGA level) - drivers of vaccination dropout by antigen transition\n"
             "LASSO on 15 LGA archetype covariates | Nigeria, DHIS2 2021-2024",
             fontsize=13, fontweight="bold", color=NAVY)
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig("fig_domain2_lasso_lga.png", dpi=180, bbox_inches="tight", facecolor="white")
pd.DataFrame(results).round(3).to_csv("domain2_lasso_coefficients_lga.csv")
print("\nTop driver per transition (largest |coef|):")
for tgt, _, _ in TARGETS:
    c = results[tgt]; top = c.abs().idxmax()
    print(f"  {tgt}: {top} ({c[top]:+.2f})")
print("wrote fig_domain2_lasso_lga.png, domain2_lasso_coefficients_lga.csv")


## 6. Domain 3 (LGA level) - composite equity-deprivation index, tiers and rankings
Equal-weight index of remoteness, low education, poverty and low wealth (each min-max 0-100), cut into
four tiers; LGA ranking, updated state ranking, and the combined priority list.

In [ ]:
"""
Domain 3 at LGA level: composite equity-deprivation index (four tiers) + rankings.
Mirrors the state-level index (Rural x HTR x Maternal no-education x Lowest wealth), rebuilt on the
richer LGA covariates: remoteness (travel time), low education, poverty, low wealth (equal weights,
each min-max 0-100, averaged, cut at 25/50/75). Outputs LGA ranking, updated STATE ranking, and a
combined priority list (zero-dose burden + equity tier + archetype).
"""
import matplotlib
matplotlib.use("Agg")
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

GEO = r"..\claude_model\data\sample\geo\nga_lgas.geojson"
XLSX = "LGA_archetype_master.xlsx"
TIER_COL = {1: "#B2182B", 2: "#EF8A62", 3: "#FDDBC7", 4: "#92C5DE"}
TIER_LAB = {1: "Tier 1: Most deprived", 2: "Tier 2: Deprived", 3: "Tier 3: Moderate", 4: "Tier 4: Least deprived"}
NAVY = "#1F3B57"


def mm(s):
    s = pd.to_numeric(s, errors="coerce")
    return (s - s.min()) / (s.max() - s.min()) * 100


cov = pd.read_excel(XLSX, "LGA covariates")
# four structural deprivation components (equal weight); higher = more deprived
comp = pd.DataFrame({
    "remoteness": mm(cov["travel_time_hc"]),
    "low_education": mm(-cov["edu_mean_years_women_15_49"]),
    "poverty": mm(cov["poverty_rate"]),
    "low_wealth": mm(-cov["relative_wealth_index"]),
})
cov["equity_deprivation_index"] = comp.mean(axis=1).round(1)
cov["equity_tier"] = pd.cut(cov["equity_deprivation_index"], [-1, 25, 50, 75, 101],
                            labels=[4, 3, 2, 1]).astype(int)   # 1 = most deprived
cov = cov.sort_values("equity_deprivation_index", ascending=False).reset_index(drop=True)
cov["equity_rank"] = range(1, len(cov) + 1)

# correlation with the modelled zero-dose (validation)
v = cov.dropna(subset=["zero_dose_rate_pct"])
r = np.corrcoef(v["equity_deprivation_index"], v["zero_dose_rate_pct"])[0, 1]
print(f"LGA composite index vs modelled zero-dose rate: Pearson r = {r:.2f} (n={len(v)})")

# ---- LGA ranking file ----
lga_rank = cov[["equity_rank", "platform_State", "platform_LGA", "equity_deprivation_index",
                "equity_tier", "archetype_type", "zero_dose_rate_pct", "zero_dose_children"]]
lga_rank.to_csv("lga_equity_ranking.csv", index=False)

# ---- combined priority list (zero-dose burden rank + equity tier + archetype) ----
comb = cov.dropna(subset=["zero_dose_children"]).copy()
comb = comb.sort_values("zero_dose_children", ascending=False).reset_index(drop=True)
comb["zero_dose_burden_rank"] = range(1, len(comb) + 1)
comb["priority_flag"] = np.where((comb["zero_dose_burden_rank"] <= 146) & (comb["equity_tier"] <= 2),
                                 "TOP PRIORITY (high burden + high deprivation)", "")
comb[["zero_dose_burden_rank", "platform_State", "platform_LGA", "zero_dose_children",
      "zero_dose_rate_pct", "equity_tier", "equity_deprivation_index", "archetype_type",
      "priority_flag"]].to_csv("lga_combined_priority_ranking.csv", index=False)

# ---- updated STATE ranking from the new LGA tiers ----
st = cov.groupby("platform_State").agg(
    mean_index=("equity_deprivation_index", "mean"),
    tier1_LGAs=("equity_tier", lambda s: int((s == 1).sum())),
    n_LGAs=("equity_tier", "size"),
    mean_zero_dose=("zero_dose_rate_pct", "mean")).reset_index()
st = st.sort_values("mean_index", ascending=False).reset_index(drop=True)
st["state_rank"] = range(1, len(st) + 1)
st["state_tier"] = pd.cut(st["mean_index"], [-1, 25, 50, 75, 101], labels=[4, 3, 2, 1]).astype(int)
st.round(1).to_csv("state_ranking_updated.csv", index=False)

# ---- figures: LGA tier choropleth + state ranking ----
g = gpd.read_file(GEO)
g["_k"] = g["state"].str.lower().str.strip() + "|" + g["lga"].str.lower().str.strip()
cov["_k"] = cov["platform_State"].str.lower().str.strip() + "|" + cov["platform_LGA"].str.lower().str.strip()
g = g.merge(cov[["_k", "equity_tier"]], on="_k", how="left")

fig, (axm, axr) = plt.subplots(1, 2, figsize=(15, 8), gridspec_kw={"width_ratios": [1.5, 1]})
g["_c"] = g["equity_tier"].map(TIER_COL).fillna("#EEEEEE")
g.plot(ax=axm, color=g["_c"].values, edgecolor="white", linewidth=0.25)
axm.set_title("LGA equity-deprivation tier, Nigeria", fontsize=13, fontweight="bold", color=NAVY, loc="left")
axm.axis("off")
axm.legend(handles=[Patch(facecolor=TIER_COL[t], label=TIER_LAB[t]) for t in TIER_COL],
           loc="lower left", fontsize=9, frameon=False)

top = st.head(25).iloc[::-1]
axr.barh(range(len(top)), top["mean_index"], color=[TIER_COL[t] for t in top["state_tier"]])
axr.set_yticks(range(len(top))); axr.set_yticklabels(top["platform_State"], fontsize=8)
axr.set_xlabel("Mean LGA equity-deprivation index (0-100)", fontsize=9)
axr.set_title("State ranking (from LGA tiers) - top 25", fontsize=12, fontweight="bold", color=NAVY, loc="left")
axr.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig_equity_tiers_lga.png", dpi=200, bbox_inches="tight", facecolor="white")

print("\nMost-deprived states (updated ranking):")
print(st.head(8)[["state_rank", "platform_State", "mean_index", "state_tier", "tier1_LGAs", "mean_zero_dose"]].round(1).to_string(index=False))
print("\nTop 10 LGAs by equity deprivation:")
print(lga_rank.head(10)[["equity_rank", "platform_State", "platform_LGA", "equity_deprivation_index", "equity_tier"]].to_string(index=False))
print("\nwrote: lga_equity_ranking.csv, lga_combined_priority_ranking.csv, state_ranking_updated.csv, fig_equity_tiers_lga.png")


## 7. Domain 3 - Multiscale Geographically Weighted Regression (MGWR), benchmarked against OLS and GWR
MGWR lets each driver's effect vary across space and reports where each covariate matters most. Compute
is heavier at 774 LGAs; run on a machine with the `mgwr` package. Outcome = modelled zero-dose rate.

In [ ]:
# MGWR / GWR / OLS benchmark (needs: pip install mgwr libpysal)
import numpy as np, pandas as pd, geopandas as gpd
from sklearn.preprocessing import StandardScaler
cov = pd.read_excel("LGA_archetype_master.xlsx", "LGA covariates").dropna(subset=["zero_dose_rate_pct"])
g = gpd.read_file(r"../claude_model/data/sample/geo/nga_lgas.geojson")
g["_k"] = g.state.str.lower().str.strip()+"|"+g.lga.str.lower().str.strip()
cov["_k"] = cov.platform_State.str.lower().str.strip()+"|"+cov.platform_LGA.str.lower().str.strip()
gg = g.merge(cov, on="_k").to_crs(3857)
DRIVERS = ["poverty_rate","edu_mean_years_women_15_49","travel_time_hc","relative_wealth_index",
           "stunting_prev_u5","conflict_fatalities"]
y = StandardScaler().fit_transform(gg[["zero_dose_rate_pct"]])
X = StandardScaler().fit_transform(gg[DRIVERS].fillna(gg[DRIVERS].median()))
coords = list(zip(gg.geometry.centroid.x, gg.geometry.centroid.y))

import statsmodels.api as sm
ols = sm.OLS(y, sm.add_constant(X)).fit(); print("OLS R2:", round(ols.rsquared,3))
try:
    from mgwr.sel_bw import Sel_BW
    from mgwr.gwr import GWR, MGWR
    bw = Sel_BW(coords, y, X).search(); gwr = GWR(coords, y, X, bw).fit()
    print("GWR bw:", bw, "| GWR R2:", round(gwr.R2,3))
    selm = Sel_BW(coords, y, X, multi=True); bws = selm.search()
    mgwr = MGWR(coords, y, X, selm).fit()
    print("MGWR bandwidths per driver:", bws, "| MGWR R2:", round(mgwr.R2,3))
    # local coefficient surfaces -> gg["b_<driver>"] for mapping
    for i, d in enumerate(DRIVERS):
        gg["b_"+d] = mgwr.params[:, i+1]
    gg.to_file("mgwr_local_coefficients.geojson", driver="GeoJSON")
    print("wrote mgwr_local_coefficients.geojson (map each b_<driver> surface)")
except Exception as e:
    print("Install mgwr+libpysal to run GWR/MGWR. OLS benchmark above. Detail:", e)

## Outputs
- `LGA_archetype_master.xlsx` - covariates + archetype + zero-dose (filterable) + Archetypes + Zero-dose tabs
- `LGA_archetypes.csv`, `LGA_archetype_profiles.csv` - assignments and profiles
- `fig_archetype_zerodose_overlay.png`, `fig_equity_tiers_lga.png`, `fig_domain2_lasso_lga.png`
- `lga_combined_priority_ranking.csv`, `lga_equity_ranking.csv`, `state_ranking_updated.csv`
- `mgwr_local_coefficients.geojson` - MGWR local coefficient surfaces

**Method note:** archetypes use predictors only; zero-dose is the outcome. Covariate vintages span
2014-2021 (DHS surfaces 2018, ACLED 2021-2024). Figures are model estimates.